# VeriGym - A quick workflow presentation

## Imports

In [1]:
import stormpy
import gymnasium as gym
import numpy as np
from pprint import pprint

import verigym
from verigym.abstraction.gym_utils.transform_observation import ReplaceInfObservation
from verigym.frameworks.stormpy.stormpy_utils import build_stormpy_mdp

# Helper function to mean over rewards
def get_mean_reward_from_trajectories(trajectories):
    return np.mean([len(traj) for traj in trajectories])

import logging
logging.getLogger().setLevel(logging.ERROR)

/Users/juleschmidt/Documents/VeriGym/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Environment

We use the  OpenAI gymnasium Cart Pole environment for this demo:

<img src="cart_pole.gif" width="400"/>


In [2]:
# 1. Load cart pole in gym
gym_env = gym.make("CartPole-v1")
print("Environment type: ", type(gym_env))

# 2. For discretization, replace (-inf, inf) observation bounds
gym_env = ReplaceInfObservation(gym_env, neg_inf=-10, pos_inf=10)
print("Observation space shape: ", gym_env.observation_space.shape)
print("Observation upper bounds: ", gym_env.observation_space.high)
print("Observation lower bounds: ", gym_env.observation_space.low)
print("Type of observation space: ", type(gym_env.observation_space))

Environment type:  <class 'gymnasium.wrappers.common.TimeLimit'>
Observation space shape:  (4,)
Observation upper bounds:  [ 4.8        10.          0.41887903 10.        ]
Observation lower bounds:  [ -4.8        -10.          -0.41887903 -10.        ]
Type of observation space:  <class 'gymnasium.spaces.box.Box'>


Now, we load the `gym_env` into our framework, turning it into a (`gym`-like) `VeriGymEnv`:

In [3]:
# Create a VeriGymEnv from gym env
generative_model = verigym.GenerativeEnv.from_gymnasium(gym_env)

print("Environment type: ", type(generative_model))
print("Observation space shape: ", generative_model.observation_space.shape)
print("Observation upper bounds: ", generative_model.observation_space.high)
print("Observation lower bounds: ", generative_model.observation_space.low)
print("Type of observation space: ", type(generative_model.observation_space))

Environment type:  <class 'verigym.environments.generativeenv.GenerativeEnv_from_gym'>
Observation space shape:  (4,)
Observation upper bounds:  [ 4.8        10.          0.41887903 10.        ]
Observation lower bounds:  [ -4.8        -10.          -0.41887903 -10.        ]
Type of observation space:  <class 'gymnasium.spaces.box.Box'>


The observation space is the same as before, but now we can apply learn the underlying model and perform any abstraction!

## 2. Abstraction

Now we: 
1. Apply a user-specified discretization to the state space
2. Learn the transition and reward functions through random search

In [4]:
abstracted_model = verigym.create_abstraction(
    original_env=generative_model,
    bin_edges_per_dim=5, 
    exploration_strategy="random", 
    num_steps=int(1e5),
)

print("Type of observation space: ", type(abstracted_model.observation_space))
print(f"The abstract observation space has {abstracted_model.observation_space.n} states!")
print("Type of environment: ", type(abstracted_model))

Type of observation space:  <class 'gymnasium.spaces.discrete.Discrete'>
The abstract observation space has 625 states!
Type of environment:  <class 'verigym.environments.explicitenv.ExplicitEnv'>


The abstract model allows us to explicitly access the model's transition and reward functions.

In [5]:
state = 156

print(f"Transitions from state {state}:")
pprint(abstracted_model.transition_function[state])
print()
print(f"State-action rewards of state {state}:")
pprint(abstracted_model.reward_function[state])


Transitions from state 156:
{0: defaultdict(<function learn_transition_function.<locals>.<lambda> at 0x11ae8bec0>,
                {151: 0.017857142857142856,
                 156: 0.17542016806722688,
                 157: 0.8067226890756303}),
 1: defaultdict(<function learn_transition_function.<locals>.<lambda> at 0x11ae8b560>,
                {151: 0.012170385395537525,
                 156: 0.1876267748478702,
                 176: 0.007099391480730223,
                 181: 0.7931034482758621})}

State-action rewards of state 156:
defaultdict(<function learn_reward_function.<locals>.<lambda>.<locals>.<lambda> at 0x12bea9c60>,
            {0: np.float64(1.0),
             1: np.float64(1.0)})


## 3. Model checking

We now want to model check the abstraction we learned in `stormpy` w.r.t maximal cumulative reward. 

Therefore, we
1. Export the model to a `stormpy.storage.SparseMdp`
2. Model check the MDP in `stormpy` and receive a scheduler (policy)

In [6]:
stormpy_mdp = build_stormpy_mdp(abstracted_model)

print(stormpy_mdp)


-------------------------------------------------------------- 
Model type: 	MDP (sparse)
States: 	625
Transitions: 	782
Choices: 	641
Reward Models:  reward0
State Labels: 	2 labels
   * deadlock -> 609 item(s)
   * init -> 16 item(s)
Choice Labels: 	none
-------------------------------------------------------------- 



In [7]:
gamma = 0.95 # discount factor
prop = stormpy.parse_properties(f"Rmax=?[Cdiscount={gamma}]")[0] # checking for cumulative discounted reward

# Run the model checking:
result = stormpy.check_model_sparse(stormpy_mdp, prop, extract_scheduler=True)
scheduler = result.scheduler

## 4. Policy evaluation

Finally, we want to evaluate our model checked policy on the original and the abstract model.

1. We convert the `stormpy` policy to a `verigym.StormpyPolicy`
2. We apply it to both the original and the abstract model.

In [8]:
# policy mapped to the original model
verigym_policy = verigym.StormpyPolicy(scheduler, abstracted_model.abstraction_map)
# policy on the abstract model
verigym_policy_on_abstracted = verigym.StormpyPolicy(scheduler, abstraction_mapper = verigym.AbstractionMapper())

n_simulation_steps = int(10e4)

In [9]:
# simulation in the original (continuous) model
trajectories_original = generative_model.simulate(
    policy=verigym_policy,
    n_steps=n_simulation_steps,
)
mean_rewards_original = get_mean_reward_from_trajectories(trajectories_original)

In [10]:
# verify the policy: (2) policy performance on abstracted model
trajectories_abstracted = abstracted_model.simulate(
    policy=verigym_policy_on_abstracted,
    n_steps=n_simulation_steps,
)
mean_rewards_abstracted = get_mean_reward_from_trajectories(trajectories_abstracted)

In [11]:
print("Comparing the mean cumulative reward of the abstract policy:")
print("Original environment: ", mean_rewards_original)
print("Abstract environment: ", mean_rewards_abstracted)

Comparing the mean cumulative reward of the abstract policy:
Original environment:  166.11295681063123
Abstract environment:  48.00768122899664
